In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("MachineLearningRating_v3.txt", sep="|", engine="python", on_bad_lines='warn')

In [ ]:
import scipy.stats as stats

claim_counts = df.groupby('Province')['TotalClaims'].apply(lambda x: (x > 0).sum())
policy_counts = df['Province'].value_counts()

# Chi-squared test
chi2, pval, _, _ = stats.chi2_contingency([claim_counts, policy_counts])
 
print(f"P-value = ",{pval})

if pval < 0.05:
    print("✅ Reject H₀: Risk differs across provinces")
else:
    print("❌ Fail to reject H₀: No evidence of risk difference across provinces") 

P-value =  {np.float64(0.0)}
✅ Reject H₀: Risk differs across provinces


In [ ]:
zip_claims = df.groupby('PostalCode')['TotalClaims'].apply(lambda x: (x > 0).mean())

stats.f_oneway(*(df[df['PostalCode'] == z]['TotalClaims'] for z in df['PostalCode'].unique() if len(df[df['PostalCode'] == z]) > 30))

print(f"P-value = ",{pval})

if pval < 0.05:
    print("✅ Reject H₀: Risk differs across zipcode")
else:
    print("❌ Fail to reject H₀: No evidence of risk difference across zipcode") 

P-value =  {np.float64(0.9701673296142395)}
❌ Fail to reject H₀: No evidence of risk difference across zipcode


In [ ]:
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
margins_by_zip = [group['Margin'] for _, group in df.groupby('PostalCode') if len(group) > 30]

print(f"P-value = ",{pval})

f_stat, pval = stats.f_oneway(*margins_by_zip)
if pval < 0.05:
    print("✅ Reject H₀: Margins differ significantly between zip codes")
else:
    print("❌ Fail to reject H₀: No significant difference in margin between zip codes")


P-value =  {np.float64(0.9701673296142395)}
❌ Fail to reject H₀: No significant difference in margin between zip codes


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

count = [(df[df['Gender'] == 'Female']['TotalClaims'] > 0).sum(),
         (df[df['Gender'] == 'Male']['TotalClaims'] > 0).sum()]
nobs = [df['Gender'].value_counts()['Female'], df['Gender'].value_counts()['Male']]

z_stat, pval = proportions_ztest(count, nobs)

print(f"P-value = ",{pval})

if pval < 0.05:
    print("✅ Reject H₀: Significant risk difference between men and women")
else:
    print("❌ Fail to reject H₀: No evidence of risk difference by gender")


P-value =  {np.float64(0.8404941485359676)}
❌ Fail to reject H₀: No evidence of risk difference by gender


Selected Metrics:

Claim Frequency= Number of policies with claims / Total Number of Policies

Claim Severity= Total Claims / Number of Policies with claims

Margin=Total Premium−Total Claims

In [20]:
group_a = df[df['Province'] == 'Gauteng']
group_b = df[df['Province'] == 'Western Cape']

claim_freq_a = (group_a['TotalClaims'] > 0).mean()
claim_freq_b = (group_b['TotalClaims'] > 0).mean()

severity_a = group_a[group_a['TotalClaims'] > 0]['TotalClaims'].mean()
severity_b = group_b[group_b['TotalClaims'] > 0]['TotalClaims'].mean()

margin_a = (group_a['TotalPremium'] - group_a['TotalClaims']).mean()
margin_b = (group_b['TotalPremium'] - group_b['TotalClaims']).mean()

In [23]:
# For Claim Frequency (binary outcome): use proportion z-test

from statsmodels.stats.proportion import proportions_ztest

claims_a = (group_a['TotalClaims'] > 0).sum()
claims_b = (group_b['TotalClaims'] > 0).sum()
nobs = [len(group_a), len(group_b)]

z_stat, pval = proportions_ztest([claims_a, claims_b], nobs)
print(f"Z-test for claim frequency → z: {z_stat:.2f}, p: {pval:.4f}")


Z-test for claim frequency → z: 7.52, p: 0.0000


In [24]:
# For Claim Severity and Margin (continuous): use t-test

from scipy.stats import ttest_ind, mannwhitneyu

severity_a = group_a[group_a['TotalClaims'] > 0]['TotalClaims']
severity_b = group_b[group_b['TotalClaims'] > 0]['TotalClaims']

# Use t-test if normally distributed
t_stat, p = ttest_ind(severity_a, severity_b, equal_var=False)
print(f"T-test for claim severity → t: {t_stat:.2f}, p: {p:.4f}")

# Or non-parametric Mann-Whitney test
u_stat, p_mw = mannwhitneyu(severity_a, severity_b)
print(f"Mann-Whitney U for claim severity → U: {u_stat}, p: {p_mw:.4f}")


T-test for claim severity → t: -2.17, p: 0.0306
Mann-Whitney U for claim severity → U: 236926.5, p: 0.3570


In [26]:
# Margin Comparison (t-test)

group_a.loc[:, 'Margin'] = group_a['TotalPremium'] - group_a['TotalClaims']
group_b.loc[:, 'Margin'] = group_b['TotalPremium'] - group_b['TotalClaims']

t_stat_margin, p_margin = ttest_ind(group_a['Margin'], group_b['Margin'], equal_var=False)
print(f"T-test for margin → t: {t_stat_margin:.2f}, p: {p_margin:.4f}")


T-test for margin → t: -1.39, p: 0.1636
